# Notebook 1: Data Exploration and Preparation for RL Intraday Trading

## Welcome to RL for Dummies! 🚀

This notebook series will teach you **Reinforcement Learning (RL) for intraday trading** from scratch.

### What you'll learn in this notebook:
1. Understanding the available intraday trading datasets
2. Exploring Level 0 market data (OHLCV)
3. Visualizing price patterns and trading opportunities
4. Understanding technical indicators
5. Preparing data for RL training

### Prerequisites:
- Basic Python knowledge
- Understanding of financial trading concepts (open, high, low, close, volume)
- No prior RL knowledge required!

Let's get started! 💪

## Step 1: Import Libraries

First, let's import the libraries we'll need:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

## Step 2: Load and Explore Available Datasets

We have 1-minute intraday data for 5 different futures contracts:
- **MCL-1m**: Micro Crude Oil Futures
- **MGC-1m**: Micro Gold Futures
- **mes-1m**: Micro E-mini S&P 500 Futures
- **ng**: Natural Gas Futures
- **si**: Silver Futures

All data is from 2020 with 1-minute frequency (intraday trading).

In [ ]:
# Define data directory
DATA_DIR = Path('./data')

# Available contracts
contracts = {
    'MCL-1m': 'Micro Crude Oil Futures',
    'MGC-1m': 'Micro Gold Futures',
    'mes-1m': 'Micro E-mini S&P 500 Futures',
    'ng': 'Natural Gas Futures',
    'si': 'Silver Futures'
}

print("📊 Available Contracts for Trading:\n")
for code, name in contracts.items():
    train_file = DATA_DIR / code / 'train.csv'
    if train_file.exists():
        df = pd.read_csv(train_file, index_col=0)
        print(f"  {code:10s} - {name:40s} | {len(df):,} training rows")
    else:
        print(f"  {code:10s} - {name:40s} | ❌ Data not found")

## Step 3: Load a Single Contract for Detailed Exploration

Let's start with **Micro Crude Oil (MCL-1m)** as our example.

In [ ]:
# Select contract to explore
CONTRACT = 'MCL-1m'  # Change this to explore different contracts

# Load train, validation, and test data
train_df = pd.read_csv(DATA_DIR / CONTRACT / 'train.csv', index_col=0)
valid_df = pd.read_csv(DATA_DIR / CONTRACT / 'valid.csv', index_col=0)
test_df = pd.read_csv(DATA_DIR / CONTRACT / 'test.csv', index_col=0)

print(f"\n📈 Loaded {contracts[CONTRACT]}")
print(f"\nData Splits:")
print(f"  Training:   {len(train_df):,} rows")
print(f"  Validation: {len(valid_df):,} rows")
print(f"  Testing:    {len(test_df):,} rows")
print(f"\n🔍 First few rows of training data:")
train_df.head()

## Step 4: Understand the Data Structure

Let's explore what information we have in each row:

In [ ]:
print("📋 Column Information:\n")
print(f"Total columns: {len(train_df.columns)}\n")

# Group columns by type
price_cols = ['open', 'high', 'low', 'close', 'adjcp']
volume_col = ['volume']
time_cols = ['datetime', 'date']
normalized_cols = [col for col in train_df.columns if col.startswith('z')]

print("📊 Raw Price Data:")
for col in price_cols:
    if col in train_df.columns:
        print(f"  - {col:10s}: Raw {col} price")

print("\n📦 Volume Data:")
for col in volume_col:
    if col in train_df.columns:
        print(f"  - {col:10s}: Trading volume")

print("\n⏰ Time Information:")
for col in time_cols:
    if col in train_df.columns:
        print(f"  - {col:10s}: Timestamp information")

print("\n🎯 Technical Indicators (Normalized):")
for col in normalized_cols[:10]:  # Show first 10
    print(f"  - {col}")
if len(normalized_cols) > 10:
    print(f"  ... and {len(normalized_cols) - 10} more")

## Step 5: Basic Statistical Summary

Let's look at the statistical properties of our data:

In [ ]:
print("📊 Statistical Summary of Price Data:\n")
price_stats = train_df[price_cols].describe()
print(price_stats)

print("\n📦 Volume Statistics:\n")
volume_stats = train_df[['volume']].describe()
print(volume_stats)

## Step 6: Visualize Price Movement

Let's visualize the price movements to understand the trading patterns:

In [ ]:
# Sample a subset for visualization (first 3 days of data)
sample_df = train_df.head(3 * 1440)  # 1440 minutes per day
sample_df['datetime'] = pd.to_datetime(sample_df['datetime'])

# Create figure with subplots
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: Price movement (OHLC)
axes[0].plot(sample_df['datetime'], sample_df['open'], label='Open', alpha=0.5, linewidth=0.8)
axes[0].plot(sample_df['datetime'], sample_df['high'], label='High', alpha=0.5, linewidth=0.8)
axes[0].plot(sample_df['datetime'], sample_df['low'], label='Low', alpha=0.5, linewidth=0.8)
axes[0].plot(sample_df['datetime'], sample_df['close'], label='Close', linewidth=1.2)
axes[0].set_title(f'{contracts[CONTRACT]} - Price Movement (First 3 Days)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price', fontsize=12)
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Plot 2: Volume
axes[1].bar(sample_df['datetime'], sample_df['volume'], width=0.0005, alpha=0.7, color='steelblue')
axes[1].set_title('Trading Volume', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Volume', fontsize=12)
axes[1].set_xlabel('Time', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 What do you see?")
print("  - Price fluctuations throughout the day")
print("  - Volume spikes at certain times (more trading activity)")
print("  - Patterns that could be exploited by RL agents!")

## Step 7: Analyze Returns Distribution

Understanding returns is crucial for RL trading:

In [ ]:
# Calculate 1-minute returns
train_df['returns'] = train_df['close'].pct_change()

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Returns distribution
axes[0].hist(train_df['returns'].dropna(), bins=100, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of 1-Minute Returns', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Returns', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Return')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Returns over time (sample)
sample_returns = train_df['returns'].head(1440 * 5).dropna()  # 5 days
axes[1].plot(sample_returns.values, linewidth=0.5, alpha=0.7)
axes[1].set_title('Returns Over Time (First 5 Days)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time (minutes)', fontsize=12)
axes[1].set_ylabel('Returns', fontsize=12)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\n📈 Returns Statistics:")
print(f"  Mean Return:        {train_df['returns'].mean():.6f}")
print(f"  Std Deviation:      {train_df['returns'].std():.6f}")
print(f"  Min Return:         {train_df['returns'].min():.6f}")
print(f"  Max Return:         {train_df['returns'].max():.6f}")
print(f"  Sharpe Ratio (Ann): {(train_df['returns'].mean() / train_df['returns'].std()) * np.sqrt(252 * 1440):.2f}")

## Step 8: Explore Technical Indicators

Technical indicators are features that will help our RL agent make decisions:

In [ ]:
# Get momentum indicators (zd_X columns)
momentum_cols = [col for col in train_df.columns if col.startswith('zd_')]

# Sample data for visualization
sample_df = train_df.head(1440 * 3).copy()  # 3 days
sample_df.index = range(len(sample_df))

# Plot momentum indicators
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: Price
axes[0].plot(sample_df.index, sample_df['close'], linewidth=1.5, color='black', label='Close Price')
axes[0].set_title('Close Price', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Momentum indicators
for col in momentum_cols:
    axes[1].plot(sample_df.index, sample_df[col], label=col, alpha=0.7, linewidth=1)

axes[1].set_title('Momentum Indicators (Price Change Signals)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time (minutes)', fontsize=12)
axes[1].set_ylabel('Momentum', fontsize=12)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].legend(loc='best', fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎯 About Technical Indicators:")
print("  - zd_5:  5-minute momentum (recent short-term trend)")
print("  - zd_10: 10-minute momentum")
print("  - zd_15: 15-minute momentum")
print("  - zd_20: 20-minute momentum")
print("  - zd_25: 25-minute momentum")
print("  - zd_30: 30-minute momentum (longer-term trend)")
print("\n  Positive values = upward momentum, Negative values = downward momentum")

## Step 9: Correlation Analysis

Let's see how different features correlate with each other:

In [ ]:
# Select key features for correlation analysis
key_features = ['close', 'volume'] + momentum_cols
corr_df = train_df[key_features].corr()

# Plot correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n💡 Key Insights:")
print("  - Momentum indicators are correlated (expected)")
print("  - Understanding correlations helps RL agents learn patterns")
print("  - Different timeframe momentums capture different trends")

## Step 10: Data Quality Check

Before training RL agents, we need to ensure data quality:

In [ ]:
print("🔍 Data Quality Report:\n")
print("="*60)

# Check for missing values
missing_values = train_df.isnull().sum()
print("\n1. Missing Values:")
if missing_values.sum() == 0:
    print("   ✅ No missing values found!")
else:
    print("   ⚠️  Found missing values:")
    print(missing_values[missing_values > 0])

# Check for infinite values
inf_values = np.isinf(train_df.select_dtypes(include=[np.number])).sum()
print("\n2. Infinite Values:")
if inf_values.sum() == 0:
    print("   ✅ No infinite values found!")
else:
    print("   ⚠️  Found infinite values:")
    print(inf_values[inf_values > 0])

# Check data continuity (time gaps)
train_df['datetime_parsed'] = pd.to_datetime(train_df['datetime'])
time_diff = train_df['datetime_parsed'].diff().dt.total_seconds()
gaps = time_diff[time_diff > 120]  # More than 2 minutes

print("\n3. Time Continuity:")
print(f"   Total records:     {len(train_df):,}")
print(f"   Time gaps found:   {len(gaps):,} (gaps > 2 minutes)")
print(f"   Expected interval: 1 minute")
if len(gaps) > 0:
    print(f"   ⚠️  Note: Gaps likely due to market hours (nights/weekends)")

# Check for duplicates
duplicates = train_df.duplicated().sum()
print("\n4. Duplicate Records:")
if duplicates == 0:
    print("   ✅ No duplicate records found!")
else:
    print(f"   ⚠️  Found {duplicates} duplicate records")

print("\n" + "="*60)
print("\n✅ Data quality check complete! Data is ready for RL training.")

## Summary and Next Steps

### 🎉 Congratulations!

You've completed Notebook 1! Here's what you learned:

1. ✅ **Loaded and explored 1-minute intraday trading data**
2. ✅ **Understood OHLCV (Open, High, Low, Close, Volume) structure**
3. ✅ **Analyzed price movements and patterns**
4. ✅ **Explored technical indicators (momentum)**
5. ✅ **Checked data quality**

### 📚 Key Concepts:
- **Level 0 Data**: Basic OHLCV market data (no order book depth)
- **Intraday Trading**: Trading within a single day using minute-level data
- **Technical Indicators**: Mathematical features derived from price data
- **Returns**: Percentage change in price (what we want to maximize!)

### 🚀 Next Steps:
In **Notebook 2**, you'll learn:
- What is Reinforcement Learning?
- How to set up a trading environment
- Training your first RL agent (DQN)
- Evaluating trading performance

Ready to train your first RL trading agent? Let's go! 🎯

---
**Pro Tip**: Try changing the `CONTRACT` variable in Step 3 to explore different futures contracts! Each has unique characteristics. 💡